> 在线讲义：[第 52 章 · 相机几何：针孔模型、内参与外参、畸变、逆透视变换、Lift-Splat-Shoot](https://wikieden.github.io/math-and-eval-stats/lessons/52-camera-geometry.html) · 本书《数学与实验统计》

# 第 52 章 · 相机几何：针孔模型、内参与外参、畸变、逆透视变换、Lift-Splat-Shoot

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/52-camera-geometry.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 投影 | K p / Z | `project(p)` | (420, 290) |
| 第二步 外参 | Rp + t | `world_to_cam(p, r, t)` | 像素 (252.15, 240) |
| 第三步 畸变 | 1 + k₁r² | `distort(xn)` | 偏移 12.5、42.2 像素 |
| 第四步 IPM | fh/(v − c_y) | `ground_from_pixel(u, v)` | 7.5 m |
| 第五步 LSS | α cᵀ | `lss_lift(logits, feat)` | 4×2 矩阵 |

例子：一台 640×480 的相机（f = 500 像素，主点 (320, 240)），装在离地 1.5 米、水平朝前的位置；径向畸变 k1 = −0.2；
地面点的逆透视变换（IPM）；Lift-Splat-Shoot 的“深度分布 × 特征”外积。
里程碑 ME-M32 的“投影 / 反投影往返测试”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
K = np.array([[500.0, 0, 320], [0, 500.0, 240], [0, 0, 1]])
H_CAM = 1.5   # 相机离地高度（米）

**函数 `project`**：针孔模型：相机坐标 (X, Y, Z) → 像素 (u, v) = (f X/Z + cx, f Y/Z + cy)。

In [ ]:
def project(p_cam: np.ndarray, k: np.ndarray = K) -> np.ndarray:
    """针孔模型：相机坐标 (X, Y, Z) → 像素 (u, v) = (f X/Z + cx, f Y/Z + cy)。"""
    uvw = k @ p_cam
    return uvw[:2] / uvw[2]

**函数 `world_to_cam`**：外参：p_cam = R p_world + t。

In [ ]:
def world_to_cam(p_world: np.ndarray, r: np.ndarray, t: np.ndarray) -> np.ndarray:
    """外参：p_cam = R p_world + t。"""
    return r @ p_world + t

**函数 `distort`**：径向畸变（归一化坐标 x = X/Z）：x_d = x(1 + k1 r² + k2 r⁴)。k1 < 0 是桶形畸变。

In [ ]:
def distort(xn: np.ndarray, k1: float = -0.2, k2: float = 0.0) -> np.ndarray:
    """径向畸变（归一化坐标 x = X/Z）：x_d = x(1 + k1 r² + k2 r⁴)。k1 < 0 是桶形畸变。"""
    r2 = (xn ** 2).sum()
    return xn * (1 + k1 * r2 + k2 * r2 * r2)

**函数 `ground_from_pixel`**：逆透视：相机水平朝前（y 轴向下），地面是 Y = h 的平面。像素反投影成射线 K⁻¹[u, v, 1]，和地面求交。

In [ ]:
def ground_from_pixel(u: float, v: float, k: np.ndarray = K, h: float = H_CAM) -> np.ndarray:
    """逆透视：相机水平朝前（y 轴向下），地面是 Y = h 的平面。像素反投影成射线 K⁻¹[u, v, 1]，和地面求交。
    返回地面点在相机坐标系里的 (X, Z)（横向、前向）。"""
    ray = np.linalg.inv(k) @ np.array([u, v, 1.0])
    s = h / ray[1]
    return np.array([s * ray[0], s * ray[2]])

**函数 `lss_lift`**：Lift-Splat-Shoot 的 lift：每个像素一个深度分布 α = softmax(logits)（D 个深度格）和一个 C 维特征 c，

In [ ]:
def lss_lift(depth_logits: np.ndarray, feat: np.ndarray) -> np.ndarray:
    """Lift-Splat-Shoot 的 lift：每个像素一个深度分布 α = softmax(logits)（D 个深度格）和一个 C 维特征 c，
    外积 α cᵀ（D×C）就是这条射线上每个深度格分到的特征。"""
    a = np.exp(depth_logits - depth_logits.max())
    a /= a.sum()
    return np.outer(a, feat)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 针孔投影：相机坐标 → 像素")
for p in ([0.0, 0.0, 5.0], [1.0, 0.5, 5.0], [1.0, 0.5, 10.0]):
    print(f"  相机坐标 {p} → 像素 {np.round(project(np.array(p)), 2).tolist()}")

In [ ]:
print("\n第二步 外参：世界 → 相机")
th = np.radians(10)
r = np.array([[np.cos(th), 0, -np.sin(th)], [0, 1, 0], [np.sin(th), 0, np.cos(th)]])
pc = world_to_cam(np.array([0.0, 0.0, 5.0]), r, np.array([0.2, 0.0, 0.0]))
print(f"  相机绕竖直轴转 10°、平移 0.2 m：世界点 (0, 0, 5) → 相机坐标 {np.round(pc, 4).tolist()} → 像素 {np.round(project(pc), 2).tolist()}")

In [ ]:
print("\n第三步 径向畸变：离中心越远偏得越多")
for xn in ([0.1, 0.0], [0.4, 0.3], [0.6, 0.45]):
    xd = distort(np.array(xn))
    print(f"  归一化坐标 {xn}（r = {np.hypot(*xn):.2f}）→ {np.round(xd, 4).tolist()}，像素偏移 {500 * np.linalg.norm(xd - xn):.1f}")

In [ ]:
print("\n第四步 逆透视：地面上的像素能直接反算出距离")
for v in (300, 340, 400, 480):
    g = ground_from_pixel(320, v)
    print(f"  像素 (320, {v})：地面前方 {g[1]:.2f} m")
print(f"  像素 (420, 340)：地面点 (横向, 前向) = {np.round(ground_from_pixel(420, 340), 3).tolist()} m")

In [ ]:
print("\n第五步 Lift-Splat-Shoot：没有深度就预测一个深度分布")
out = lss_lift(np.array([0.0, 2.0, 1.0, -1.0]), np.array([1.0, -0.5]))
print(f"  4 个深度格的 logits [0, 2, 1, −1]、特征 [1, −0.5]：每格分到的特征\n{np.round(out, 4)}")

## 3. 代码里才看得到的

1. **Y 轴朝下**：相机坐标系的约定是 x 右、y 下、z 前，所以地面是 Y = +h，v 越大（越往图像下方）越近。
2. **畸变作用在归一化坐标上**：先除以 Z 得到 (x, y)，畸变后再乘 f、加主点；k 的数值和焦距无关。
3. **没有做投影 / 反投影的往返测试**：给定深度把像素反投影回 3D、再投影回来对数，留给自己做（里程碑 ME-M32）。

## 4. 自己改一改

1. 把相机向下俯 10°（绕 x 轴转），重写 `ground_from_pixel`，同一个像素的地面距离变成多少？
2. 给 `distort` 加上 k₂ = 0.05，r = 0.75 那个点的偏移变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 针孔模型除深度乘焦距，单个像素只剩一条射线；
> 外参搬到相机系，P 等于 K 乘 [R | t]；
> 径向畸变 1 加 k₁r²，边缘偏得多；
> 已知在地面就能求距离，远处一像素差几米；
> 没有深度就预测分布，外积摊到射线上。